# Correspondence Analysis (CA): periods, gender, country, occupation, field


## Documentation
* [Correspondence Analysis](https://en.wikipedia.org/wiki/Correspondence_analysis) (Wikipedia)
* [CA - Correspondence Analysis in R: Essentials](https://www.sthda.com/english/articles/31-principal-component-methods-in-r-practical-guide/113-ca-correspondence-analysis-in-r-essentials/)

* [Analyse factorielle des correspondances](https://fr.wikipedia.org/wiki/Analyse_factorielle_des_correspondances) (Wikipedia)
* [Analyse factorielle des correspondances](https://github.com/Sciences-historiques-numeriques/histoire_numerique_methodes/blob/main/statistiques_descriptives/analyse_factorielle_correspondances_manuels.ipynb)


### Add fanalysis library to pip environment

* Open a terminal and activate your data analysis environment:

    -> my_venvs_activate data_analysis
* Install the missing package in the active environment:

    -> pip install fanalysis
* Deactivate the environment when you have finished:

    -> my_venvs_deactivate


In [ ]:
### Activate libraries that will be used in the notebook

import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

import fanalysis.ca as fa 


import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.ticker as ticker
from matplotlib.ticker import ScalarFormatter



In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')


In [ ]:
import bivariate_library as bl
import correspondence_analysis_library as cal

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Create a dataframe with the data to be analysed

In this notebook, we use the data produced in the da3-1 notebook: a list of individuals with three qualitative variables — coded gender, country of birth and period of activity.

The CA-based analysis results produced in this notebook are therefore based on the same data and complement the bivariate analysis carried out in notebook da3-1.

Of course, you can use regions instead of countries or other territories, as in notebook da3. The important point is to use the same data as in the bivariate analysis notebook to observe the differences between the two approaches. 



In [ ]:
csv_address='../da_data/da4-AFC.csv'
df_p = pd.read_csv(csv_address)
df_p.head(3)

In [ ]:
df_p = df_p.drop(['CNTR_ID', 'CNTR_NAME'], axis=1)

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p.info()

In [ ]:
df_p=df_p.rename(columns={'uriPer': 'person_uri'})

In [ ]:
df_p.head(1)

In [ ]:
pd.set_option('display.max_columns', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## CA : Factor analysis

Cf. documentation at the top of the notebook


NB We use self defined functions stored in these files:
* notebooks_jupyter/correspondence_analysis_library.py
* notebooks_jupyter/bivariate_library.py

In [ ]:
### Option that makes all the columns visible in a notebook
pd.set_option('display.max_columns', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## Gender vs country

In [ ]:
def code_gender(gender: str):
    if gender == 'female':
        output='f'
    else:
        output='m'
    return output    

In [ ]:
### Apply function and create new column
df_p['per_gender']= df_p.apply(lambda x: x.periodsActivity +'_'+ code_gender(x.gender), axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
gen_vs_country=pd.crosstab(df_p.gender, df_p.coded_country, margins=True)

gen_vs_country

In [ ]:
observed = gen_vs_country.iloc[:-1, :-1 ]

### Chi-square test

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# Represent dimension 1 and 2
cal.print_eigenvalue(afc)

In [ ]:
# Represent dimension 1 and 1
# Normally we would represent this on the horizontal axis,
# but this is not possible in the fanalysis library,
#cf. below
afc.mapping(num_x_axis=1,num_y_axis=1,figsize=(8,8))
plt.savefig('images/DA4_Gender_vs_Country.jpg')

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(12, 4))
plt.savefig('images/da4heatmap_Gender_vs_Country.jpg')

Commentaire :
* We observe that Spain shows a strong statistical bias toward the female gender, while the Russia/Belarus region shows a marked bias toward the male gender.

* We also observe that regions such as Germany or the Global South have residual values close to zero, indicating that they do not deviate significantly from a random distribution between genders, unlike the extremes identified previously.

The reason for this observation is in the way the CA code calculates the proportion of the contribution of these categories to the variation,and the representation is not linear, or depending on the position of all the other categories.


In [ ]:
observed_f = observed #.loc[:, [col for col in observed.columns if col not in ['professor']]]

In [ ]:
observed_f=observed_f.reindex(sorted(observed_f.columns, reverse=True), axis=1)

In [ ]:
# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 12))

colors = plt.cm.Set2.colors
lefts = np.zeros(len(observed_f.columns))  # horizontal accumulator

for i, (row_label, row_vals) in enumerate(observed_f.iterrows()):
    ax.barh(
        observed_f.columns,
        row_vals,
        left=lefts,
        color=colors[i % len(colors)],
        label=row_label,
        edgecolor='white',
        linewidth=0.6,
    )
    # Label inside each segment
    for j, (col, val) in enumerate(zip(observed_f.columns, row_vals)):
        if val > 50:
            ax.text(
                lefts[j] + val / 2, j,
                f'{int(val):,}',
                ha='center', va='center',
                fontsize=7, color='white', fontweight='bold'
            )
    lefts += row_vals.values

# --- Fix scientific notation ---
fmt = ScalarFormatter()
fmt.set_scientific(False)
fmt.set_useOffset(False)
ax.xaxis.set_major_formatter(fmt)  # x-axis now holds the counts



# --- Formatting ---
ax.set_xlabel('Number')
ax.set_title('Number of persons per occupation class and period', fontsize=12)
ax.legend(title='Activity Periods', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
observed

### Advanced observation of coordinates, contribution and cos2 (best represented)

In [ ]:
dfc=pd.DataFrame([list(afc.col_coord_.round(3)),list(afc.col_contrib_.round(3)),list(afc.col_cos2_.round(3))])
dfc.columns=observed.columns
dfc.index=['coord.', 'contrib.', 'cos2 [repr. quality]']
dfc

In [ ]:
print(round(float(afc.col_contrib_.sum()), 2))

In [ ]:
dfr=pd.DataFrame([list(afc.row_coord_.round(3)),list(afc.row_contrib_.round(3))])
dfr.columns=observed.index
dfr.index=['coord.', 'contrib.']
dfr

In [ ]:
###  fanalysis coord.

row_coords_fan = pd.DataFrame(afc.row_coord_,
                               index=observed.index,
                               columns=[f"Dim{i+1}" for i in range(afc.row_coord_.shape[1])])

col_coords_fan = pd.DataFrame(afc.col_coord_,
                               index=observed.columns,
                               columns=[f"Dim{i+1}" for i in range(afc.col_coord_.shape[1])])


In [ ]:
# Usage with fanalysis
cal.plot_ca_single_axis(row_coords_fan, col_coords_fan,
                    title="CA fanalysis— Gender vs 8 categories (Dim 1)")


## Gender and period vs country

In [ ]:
### Apply function and create new column
df_p['per_gender']= df_p.apply(lambda x: x.periodsActivity +'_'+ code_gender(x.gender), axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
perGen_vs_country=pd.crosstab(df_p.per_gender, df_p.coded_country, margins=True)

perGen_vs_country

In [ ]:
observed = perGen_vs_country.iloc[:-1, :-1 ]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
#data preparation for the chi-square test
### 1. copy the function and data
df_group = perGen_vs_country.copy()

# 2. new grouping of countries
df_group['Western Europe'] = df_group['United Kingdom'] + df_group['Western Europe'] + df_group['France'] + df_group['Germany'] + df_group['Spain']
df_group['Eastern Europe'] = df_group['Ukraine'] + df_group['Russian Fed. Belarus'] + df_group['Poland'] + df_group['Central/Eastern Europe']
df_group['Rest of World'] = df_group['Global South'] + df_group['Asia-Pacific'] + df_group['Middle East']
df_group['North America'] = df_group['United States Can.']

# 3. new grouping of periods
df_group.loc['1846-1945_f'] = df_group.loc[['1871-1895_f', '1896-1920_f', '1921-1945_f']].sum()
df_group.loc['1946-1995_f'] = df_group.loc[['1946-1970_f', '1971-1995_f']].sum()
df_group.loc['1996-2045_f'] = df_group.loc[['1996-2020_f', '2021-2045_f']].sum()

df_group.loc['1846-1945_m'] = df_group.loc[['1846-1870_m', '1871-1895_m', '1896-1920_m', '1921-1945_m']].sum()
df_group.loc['1946-1995_m'] = df_group.loc[['1946-1970_m', '1971-1995_m']].sum()
df_group.loc['1996-2045_m'] = df_group.loc[['1996-2020_m', '2021-2045_m']].sum()

# 4. creation of the new table
colonnes_finales = ['Western Europe', 'Eastern Europe', 'North America', 'Rest of World']
lignes_finales = ['1846-1945_m','1846-1945_f', '1946-1995_m','1946-1995_f', '1996-2045_m', '1996-2045_f']

observed = df_group.loc[lignes_finales, colonnes_finales]

observed

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)
plt.savefig('images/dim_contributions_countriesVSgenderVSperiods.jpg')

In [ ]:
df = afc.col_topandas()[['col_cos2_dim1',
                            'col_cos2_dim2',
                            'col_cos2_dim3']].sort_values(by=['col_cos2_dim1'], ascending=False)
df

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(20,10))
plt.savefig('images/dim_1and2_countriesVSgenderVSperiods.jpg')

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2, num_y_axis=3, figsize=(20,10))

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(12, 6))

In [ ]:
### do not forget the number:
observed

## Analyse countries and activity periods

In [ ]:
### Inspect coded countries
print(df_p.groupby(by='coded_country').size().sort_values(ascending=False))

In [ ]:
### Inspect activity periods
print(df_p.groupby(by='periodsActivity').size().sort_values(ascending=False))

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_vs_country=pd.crosstab(df_p.periodsActivity, df_p.coded_country, margins=True)

## display all columns
pd.set_option('display.max_columns', None)

per_vs_country

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
# 1. Countries
def regrouper_coded_country (coded_country):
    if coded_country in ['United Kingdom', 'Western Europe', 'France', 'Germany', 'Spain']:
        output= 'Western Europe'
    elif coded_country in ['Ukraine', 'Russian Fed. Belarus', 'Poland', 'Central/Eastern Europe']:
        output= 'Eastern Europe'
    elif coded_country in ['Global South', 'Asia-Pacific', 'Middle East']:
        output= 'Rest of World'
    elif coded_country in ['United States Can.']:
        output= 'North America'
    else:
        output=coded_country
    return output
# 2. Periods
def regrouper_periodsActivity(periodsActivity):
    if '1846-1870' in periodsActivity or '1871-1895' in periodsActivity or '1896-1920' in periodsActivity:
            return '1846-1920'
    elif '1996-2020' in periodsActivity or '2021-2045' in periodsActivity:
            return '1996-2045'
    else:
        output=periodsActivity
    return output

# Application at the function
df_p['coded_country'] = df_p['coded_country'].apply(regrouper_coded_country)
df_p['periodsActivity'] = df_p['periodsActivity'].apply(regrouper_periodsActivity)

# cross table
per_vs_country = pd.crosstab(
    df_p['periodsActivity'],
    df_p['coded_country'],
    margins=True
)

per_vs_country

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(12, 4))

### Inspect the number of individuals, the real distribution

In [ ]:
observed_f = observed# .loc[:, [col for col in observed.columns if col not in ['professor']]]

In [ ]:
observed_f=observed_f.reindex(sorted(observed_f.columns, reverse=True), axis=1)

In [ ]:
# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 12))

colors = plt.cm.Set2.colors
lefts = np.zeros(len(observed_f.columns))  # horizontal accumulator

for i, (row_label, row_vals) in enumerate(observed_f.iterrows()):
    ax.barh(
        observed_f.columns,
        row_vals,
        left=lefts,
        color=colors[i % len(colors)],
        label=row_label,
        edgecolor='white',
        linewidth=0.6,
    )
    # Label inside each segment
    for j, (col, val) in enumerate(zip(observed_f.columns, row_vals)):
        if val > 50:
            ax.text(
                lefts[j] + val / 2, j,
                f'{int(val):,}',
                ha='center', va='center',
                fontsize=7, color='white', fontweight='bold'
            )
    lefts += row_vals.values

# --- Fix scientific notation ---
fmt = ScalarFormatter()
fmt.set_scientific(False)
fmt.set_useOffset(False)
ax.xaxis.set_major_formatter(fmt)  # x-axis now holds the counts



# --- Formatting ---
ax.set_xlabel('Number')
ax.set_title('Number of persons per per macro-region and period/gender', fontsize=12)
ax.legend(title='Activity Periods', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

### CA (correspondence analysis)

In [ ]:
observed

In [ ]:
# Initialise the CA object, then fit (=calculate)
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square):  0.085
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(15,10))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(15,15))

In [ ]:
## Use the adjusted residual to test the interpretation
pp = bl.plot_chi2_residuals(observed, figsize=(10, 4))

## Occupation and field as additional qualitative variables

In [ ]:
csv_address='../da_data/da4-persons-features.csv'
# Explicitly tell pandas which strings to treat as NA (exclude 'NA' from the list)
# By default, pandas treats 'NA', 'N/A', 'NaN', etc. as missing values.
# If you want to override this by specifying a custom list that does NOT include 'NA'.
## df_pfeat = pd.read_csv(csv_address, na_values=['', 'N/A', 'NULL', 'None'])
df_pfeat = pd.read_csv(csv_address)
df_pfeat.head()

In [ ]:
df_pfeat = df_pfeat.drop(columns=['field_sec2'])

In [ ]:
### Observe the 
df_pfeat.info()

In [ ]:
df_p = pd.merge(df_p, df_pfeat, on='person_uri', how='left')

In [ ]:
df_p.info()

In [ ]:
df_p.iloc[50:54]

In [ ]:
### missing values are excluded by default
# To keep them add parameter , dropna=False
df_p.groupby(by='occupation_main').size()

## Country vs main occupation

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
occupation_vs_country=pd.crosstab(df_p.occupation_main, df_p.coded_country, margins=True)

occupation_vs_country

In [ ]:
observed = occupation_vs_country.iloc[:-1, :-1 ]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# One only dimension represents the whole inertia/variance
cal.print_eigenvalue(afc)

In [ ]:
###  fanalysis coord.

row_coords_fan = pd.DataFrame(afc.row_coord_,
                               index=observed.index,
                               columns=[f"Dim{i+1}" for i in range(afc.row_coord_.shape[1])])

col_coords_fan = pd.DataFrame(afc.col_coord_,
                               index=observed.columns,
                               columns=[f"Dim{i+1}" for i in range(afc.col_coord_.shape[1])])


In [ ]:
# Usage with fanalysis
cal.plot_ca_single_axis(row_coords_fan, col_coords_fan,
                    title="CA fanalysis— Gender vs 8 categories (Dim 1)")


In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(12, 4))
plt.savefig('images/Country_vs_main_occupation.jpg')

## Period vs main occupation

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
occupation_vs_activityPeriod=pd.crosstab(df_p.occupation_main, df_p.periodsActivity, margins=True)
occupation_vs_activityPeriod

In [ ]:
observed = occupation_vs_activityPeriod.iloc[:-1, :-1 ]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# Represent dimension 1 and 2
cal.print_eigenvalue(afc)

In [ ]:
###  fanalysis coord.

row_coords_fan = pd.DataFrame(afc.row_coord_,
                               index=observed.index,
                               columns=[f"Dim{i+1}" for i in range(afc.row_coord_.shape[1])])

col_coords_fan = pd.DataFrame(afc.col_coord_,
                               index=observed.columns,
                               columns=[f"Dim{i+1}" for i in range(afc.col_coord_.shape[1])])

In [ ]:
# Usage with fanalysis
cal.plot_ca_single_axis(row_coords_fan, col_coords_fan,
                    title="CA fanalysis— Gender vs 8 categories (Dim 1)")
plt.savefig('images/Period_vs_main_occupation.jpg')

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(9, 3))

## Period and country vs main occupation

In [ ]:
### Apply function and create new column
df_p['per_occupation']= df_p.apply(lambda x: np.nan if pd.isna(x.occupation_main) \
                              else x.periodsActivity +'_'+ x.occupation_main, axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_occ_vs_country=pd.crosstab(df_p.per_occupation, df_p.coded_country, margins=True)
per_occ_vs_country

In [ ]:
observed = per_occ_vs_country.iloc[:-1, :-1 ]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
### We define a function that codes and aggregates the values in order to avoid dispersion, 
# which would make them difficult to analyse.
# 1. countries
def regrouper_coded_country (coded_country):
    if coded_country in ['United Kingdom', 'Western Europe', 'France', 'Germany', 'Spain']:
        output= 'Western Europe'
    elif coded_country in ['Ukraine', 'Russian Fed. Belarus', 'Poland', 'Central/Eastern Europe']:
        output= 'Eastern Europe'
    elif coded_country in ['Global South', 'Asia-Pacific', 'Middle East']:
        output= 'Rest of World'
    elif coded_country in ['United States Can.']:
        output= 'North America'
    else:
        output=coded_country
    return output

# 2. occupations
def regrouper_per_occ(per_occ):
    if pd.isna(per_occ):
        return 'Unknown'
    if '1846-1870_war-correspondent' in per_occ or '1871-1895_war-correspondent' in per_occ or '1896-1920_war-correspondent' in per_occ:
            return '1846-1920_war-correspondent'
    elif '1846-1870_war-photographer' in per_occ or '1871-1895_war-photographer' in per_occ or '1896-1920_war-photographer' in per_occ:
            return '1846-1920_war-photographer'
    elif '1996-2020_war-correspondent' in per_occ or '2021-2045_war-correspondent' in per_occ:
            return '1996-2045_war-correspondent'
    elif '1996-2020_war-photographer' in per_occ or '2021-2045_war-photographer' in per_occ:
            return '1996-2045_war-photographer'
    else:
        output=per_occ
    return output

# Application at the function
df_p['coded_country'] = df_p['coded_country'].apply(regrouper_coded_country)
df_p['per_occupation'] = df_p['per_occupation'].apply(regrouper_per_occ)

# cross table
per_occ_vs_country = pd.crosstab(
    df_p['per_occupation'],
    df_p['coded_country'],
    margins=True
)

per_occ_vs_country

In [ ]:
observed = per_occ_vs_country.iloc[:-2, :-1 ]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(18,14))
plt.savefig('images/DIMENSIONS1&2_Periodandcountryvsmainoccupation).jpg')

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(15,10))

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(16, 10))

## Gender and main occupation vs period

In [ ]:
### Apply function and create new column
df_p['occup_gen']= df_p.apply(lambda x: np.nan if pd.isna(x.occupation_main) \
                              else x.occupation_main +'_'+ code_gender(x.gender), axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_vs_occGen=pd.crosstab(df_p.periodsActivity, df_p.occup_gen, margins=True)
per_vs_occGen

In [ ]:
observed = per_vs_occGen.iloc[3:-1, :-1]

In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,6))

In [ ]:
pp = bl.plot_chi2_residuals(observed, figsize=(5, 5))

#### Comment: 
* Starting in 1996, there has been a significant and undeniable statistical overrepresentation of women in the role of press correspondents. The 21st century has seen the emergence and rise of female reporters in the field of journalism.
* Unlike text-based journalists, who have undergone a radical demographic and sociological transformation, the profile of photographers has remained much more stable and homogeneous over the course of the century.

In [ ]:
observed

## Secondary occupation

In [ ]:
### Inspect secondary occupations
print(df_p.groupby(by='occupation_sec1').size().sort_values(ascending=False).iloc[:50])

Commentaire : 
* It is clear that these secondary careers are often related to the fields of journalism or photography.
* The ratio is approximately 4 to 1 (617 / 144 ≈ 4.3). This indicates a very marked historical predominance of text over images in the corpus.

In [ ]:
### Inspect secondary occupations
print(df_p.groupby(by='occupation_sec2').size().sort_values(ascending=False).iloc[:50])

Commentaire : 
* The “occup_sec2” column is much more diverse, but will be analyzed later.

In [ ]:
### We define a function that codes and aggregates the values in order to avoid dispersion

def codeSecOccupation(occupation: str):
    if 'director' in occupation:
        output='writer'
    else:
        output=occupation
    return output  

In [ ]:
df_p['codedSecOccupation']=df_p.apply(lambda x : np.nan if pd.isna(x.occupation_sec1) \
                              else codeSecOccupation(x.occupation_sec1), axis=1)

In [ ]:
### Inspect secondary occupations
dfca = df_p.groupby(by='codedSecOccupation').size().sort_values(ascending=False)

In [ ]:
lc=dfca.iloc[:14].index.to_list()
print(lc)

## Secondary occupation vs country

In [ ]:
### Contingency table: 
dfc = df_p[df_p.codedSecOccupation.isin(lc)]
occ_vs_country=pd.crosstab(dfc.coded_country, dfc.codedSecOccupation, margins=True)
occ_vs_country

Commentaire
* Based on the data, it was not deemed necessary to continue the research.

## Secondary occupation vs period

In [ ]:
### Contingency table: 
dfc = df_p[df_p.codedSecOccupation.isin(lc)]
occ_vs_per=pd.crosstab(dfc.periodsActivity, dfc.codedSecOccupation, margins=True)
occ_vs_per

Commentaire
* Based on the data, it was not deemed necessary to continue the research.